# 02 — MM-Fit to the common 17-joint format

MM-Fit already contains 2D pose estimates, so this notebook **does not download the 40 GB RGB-D video package and does not rerun MediaPipe**. It downloads the official compact archive, cuts each continuous workout into labeled exercise sets, normalizes the 17 COCO joints, and downloads a ZIP compatible with notebook 03.

MM-Fit provides workout IDs (`w00`…`w20`). We preserve those as group/person identifiers so one participant cannot leak across training and test splits.


In [ ]:
from pathlib import Path
import csv
import hashlib
import os
import shutil
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from google.colab import files
from tqdm.auto import tqdm

MMFIT_URL = "https://s3.eu-west-2.amazonaws.com/vradu.uk/mm-fit.zip"
ARCHIVE_PATH = Path("/content/mm-fit.zip")
EXTRACT_ROOT = Path("/content/mm-fit-source")
OUTPUT_ROOT = Path("/content/calorie_mmfit_landmarks_v1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Free temporary disk:")
os.system("df -h /content | tail -1")

In [ ]:
if not ARCHIVE_PATH.exists():
    urllib.request.urlretrieve(MMFIT_URL, ARCHIVE_PATH)
if not EXTRACT_ROOT.exists():
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(EXTRACT_ROOT)

pose_files = sorted(EXTRACT_ROOT.rglob("w*_pose_2d.npy"))
print("2D pose files:", len(pose_files))
print("First pose:", pose_files[0] if pose_files else "NOT FOUND")
assert pose_files, 'Official archive layout changed or extraction failed.'

## Conversion logic

MM-Fit stores continuous workouts. Its label CSV gives `start frame, end frame, repetitions, exercise`. We cut one NPZ clip for every labeled exercise set. MM-Fit 2D pose uses COCO's 17-joint order, matching the selected MediaPipe joint order from notebook 01.


In [ ]:
CLASS_MAP = {
    "squats": "squat",
    "lunges": "lunge",
    "bicep_curls": "bicep_curl",
    "situps": "sit_up",
    "pushups": "push_up",
    "tricep_extensions": "tricep_extension",
    "dumbbell_rows": "dumbbell_row",
    "jumping_jacks": "jumping_jack",
    "dumbbell_shoulder_press": "shoulder_press",
    "lateral_shoulder_raises": "lateral_raise",
}
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def label_path_for(pose_path):
    direct = pose_path.with_name(pose_path.name.replace("_pose_2d.npy", "_labels.csv"))
    if direct.exists():
        return direct
    workout_id = pose_path.name.split("_")[0]
    matches = list(pose_path.parent.rglob(f"{workout_id}_labels.csv"))
    if not matches:
        matches = list(EXTRACT_ROOT.rglob(f"{workout_id}_labels.csv"))
    return matches[0] if matches else None


def unpack_pose_2d(array):
    # Official shape is (2, time, 18): frame index + 17 x/y joint values.
    if array.ndim != 3:
        raise ValueError(f"Unexpected pose rank: {array.shape}")
    if array.shape[0] == 2 and array.shape[2] >= 18:
        frame_numbers = array[0, :, 0].astype(int)
        xy = np.transpose(array[:, :, 1:18], (1, 2, 0))
    elif array.shape[-1] == 2 and array.shape[1] >= 17:
        frame_numbers = np.arange(array.shape[0])
        xy = array[:, :17, :]
    else:
        raise ValueError(f"Unsupported MM-Fit pose shape: {array.shape}")
    return frame_numbers, xy.astype(np.float32)


def normalize_2d(xy):
    result = []
    for frame in xy:
        hip_center = (frame[LEFT_HIP] + frame[RIGHT_HIP]) / 2.0
        shoulder_center = (frame[LEFT_SHOULDER] + frame[RIGHT_SHOULDER]) / 2.0
        shoulder_width = np.linalg.norm(frame[LEFT_SHOULDER] - frame[RIGHT_SHOULDER])
        torso_length = np.linalg.norm(shoulder_center - hip_center)
        scale = max(float(shoulder_width), float(torso_length), 1e-6)
        normalized_xy = (frame - hip_center) / scale
        z = np.zeros((17, 1), dtype=np.float32)
        visibility = np.ones((17, 1), dtype=np.float32)
        result.append(np.concatenate([normalized_xy, z, visibility], axis=1))
    return np.asarray(result, dtype=np.float32)


def read_labels(path):
    with path.open(newline="") as handle:
        for row in csv.reader(handle):
            yield int(row[0]), int(row[1]), int(row[2]), row[3]

In [ ]:
rows = []
for pose_path in tqdm(pose_files, desc="Converting MM-Fit workouts"):
    workout_id = pose_path.name.split("_")[0]
    labels_path = label_path_for(pose_path)
    if labels_path is None:
        print("Missing labels:", pose_path)
        continue

    frame_numbers, xy = unpack_pose_2d(np.load(pose_path))
    for set_index, (start_frame, end_frame, repetitions, source_class) in enumerate(read_labels(labels_path)):
        if source_class not in CLASS_MAP:
            continue
        mask = (frame_numbers >= start_frame) & (frame_numbers <= end_frame)
        if mask.sum() < 8:
            continue

        canonical_class = CLASS_MAP[source_class]
        landmarks = normalize_2d(xy[mask])
        clip_id = hashlib.sha1(
            f"{workout_id}:{start_frame}:{end_frame}:{source_class}".encode()
        ).hexdigest()[:16]
        output_path = OUTPUT_ROOT / canonical_class / f"{clip_id}.npz"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        timestamps_s = np.arange(len(landmarks), dtype=np.float32) / 30.0
        np.savez_compressed(
            output_path,
            landmarks=landmarks,
            timestamps_s=timestamps_s,
            label=canonical_class,
            source_dataset="mm-fit",
            original_path=f"{workout_id}:{start_frame}-{end_frame}",
            target_fps=30,
            repetitions=repetitions,
        )
        rows.append({
            "clip_id": clip_id,
            "source_dataset": "mm-fit",
            "person_id": workout_id,
            "group_id": f"mm-fit:{workout_id}",
            "source_class": source_class,
            "canonical_class": canonical_class,
            "original_path": f"{pose_path.relative_to(EXTRACT_ROOT)}:{start_frame}-{end_frame}",
            "output_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "status": "ok",
            "source_fps": 30,
            "sampled_frames": len(landmarks),
            "detection_rate": 1.0,
            "duration_s": len(landmarks) / 30.0,
            "repetitions": repetitions,
        })

manifest = pd.DataFrame(rows)
manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
print("Saved sets:", len(manifest))
print("People/workouts:", manifest["person_id"].nunique())
print()
print("Class counts:")
print(manifest["canonical_class"].value_counts().sort_index())
manifest.head()

In [ ]:
npz_files = list(OUTPUT_ROOT.rglob("*.npz"))
assert len(npz_files) == len(manifest) and len(manifest) > 0
assert manifest["person_id"].nunique() >= 15, "Too few workouts found; inspect archive extraction."

zip_path = shutil.make_archive(
    "/content/calorie_mmfit_landmarks_v1", "zip", root_dir=OUTPUT_ROOT
)
print("ZIP ready:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)